In [ ]:
import pandas as pd
import numpy as np

# PPMI

## Preprocessing

### Prepare metadata

In [ ]:
from create_master_metadata import create_master_metadata

master_metadata = create_master_metadata(
    rnaseq_meta_path='../../metadata.csv',
    proteomics_meta_path='../../proteomics/data/proteomics_metadata.csv',
    output_path='../data/metadata_m0_PPMI.csv',
    cohort="PPMI"
)

In [ ]:
# Checkpoint
meta_ppmi = pd.read_csv('../data/metadata_m0_PPMI.csv')

### Prepare expression data

### Preprocess RNAseq data

In [ ]:
from preprocess_rnaseq import preprocess_rnaseq

In [ ]:
log_matrix = preprocess_rnaseq(
    counts_file="../data/ppmi_rnaseq.csv",
    metadata_file="../data/master_baseline_metadata.csv",
    output_file="../data/ppmi_rnaseq_log1p_filtered.csv",
    visit_filter=0.0,
    cohort_name="PPMI"
)

#### QC checks for preprocessing step

In [ ]:
# Load your generated log1p matrix
log_df = pd.read_csv("../data/ppmi_rnaseq_log1p_filtered.csv", index_col=0)

In [ ]:
# Check for NaNs or infinities
print("Any NaN values?", log_df.isna().sum().sum())
print("Any Infinite values?", np.isinf(log_df.select_dtypes(include=np.number)).sum().sum())
print(f"Matrix shape (Genes x Samples): {log_df.shape}")

In [ ]:
print(log_df.iloc[:5, :5]) # Preview top-left corner
print("Overall Mean:", log_df.values.mean())
print("Overall Std Dev:", log_df.values.std())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Transpose so samples are rows for PCA (Samples x Genes)
pca_data = log_df.T

pca = PCA(n_components=5)
pcs = pca.fit_transform(pca_data)
explained_var = pca.explained_variance_ratio_ * 100
print(f"PC1 ({explained_var[0]:.1f}%)", f"PC2 ({explained_var[1]:.1f}%)")

pca_df = pd.DataFrame(pcs, index=pca_data.index)

# Merge with metadata to color-code by technical factors (e.g., plate)
m = meta.loc[meta['study']=='PPMI'].set_index('participant_id')[['plate', 'sex', 'diagnosis']]

pca_df = pca_df.join(m, how='inner')

# Plot PCA colored by sequencing plate
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x=pca_df.columns[0], y=pca_df.columns[1], hue='plate', alpha=0.7, palette='tab10', legend=False)
plt.title("PCA of Log1p Matrix (Pre-Batch Correction, colored by plate)")
plt.xlabel(f"PC1 ({explained_var[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained_var[1]:.1f}%)")
#plt.savefig( "../figures/ppmi_log1p_preBatch_byPlate_PCA1.jpg", dpi=600,bbox_inches="tight")
plt.show()


### RNA-seq technical batch correction

In [ ]:
! ./adjust_batch_effects.R \
--expression ../data/ppmi_rnaseq_log1p_filtered.csv \
--metadata ../data/master_baseline_metadata.csv \
--output ../data/ppmi_rnaseq_adjusted.csv \
--plot_prefix ../figures/ppmi_diagnostic_plots

### Proteomics

In [ ]:
from proteomics_qc import verify_proteomics_scaling, normalize_proteomics

In [ ]:
csfp_df = pd.read_csv("../data/ppmi/raw/ppmi_csf_prot.csv", index_col=0)
plap_df = pd.read_csv("../data/ppmi/raw/ppmi_pla_prot.csv", index_col=0)

In [ ]:
csf_vars = verify_proteomics_scaling(csfp_df, "CSF Proteomics")

In [ ]:
plasma_vars = verify_proteomics_scaling(plap_df, "Plasma Proteomics")

In [ ]:
# Apply normalization (log2 + median centering)
csfp_clean = normalize_proteomics(csfp_df, log2_transform=True)
plap_clean = normalize_proteomics(plap_df, log2_transform=True)

## MOFA training

### MOFA prep

In [ ]:
from prep_utils import validate_and_intersect_views, run_preflight_check

In [ ]:
ppmi_meta = pd.read_csv("../data/metadata_m0_PPMI.csv", index_col = 0)

In [ ]:
ppmi_meta = ppmi_meta[~ppmi_meta.index.duplicated(keep="first")]

In [ ]:
log_df_adjusted = pd.read_csv("../data/ppmi_rnaseq_adjusted.csv", index_col=0) 

In [ ]:
aligned_views, aligned_meta, mofa_entry_df = validate_and_intersect_views(
    rna_df=log_df_adjusted,
    csf_df=csfp_clean,
    plasma_df=plap_clean,
    metadata_df=ppmi_meta,
    output_dir="../ppmi_mofa_inputs"
)

### Train MOFA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from train_mofa import train_mofa_with_seeds

In [ ]:
train_mofa_with_seeds("../ppmi_mofa_inputs/ppmi_mofa_input_long.csv", "../ppmi_mofa_outputs", n_seeds=5, n_factors=10)

In [ ]:
python scripts/select_best_model.py --model_dir ppmi_mofa_outputs/     

# OUTPUT:
#Found 19 total file(s) in 'ppmi_mofa_outputs/'
#Found 5 matching 'model_seed_*.hdf5' file(s)
#model_seed_1.hdf5: Final ELBO = -5111028.235487757
#model_seed_2.hdf5: Final ELBO = -5111052.360271131
#model_seed_4.hdf5: Final ELBO = -5111032.848045845
#model_seed_3.hdf5: Final ELBO = -5111020.391811946
#model_seed_0.hdf5: Final ELBO = -5111026.60686289
#
#Best model identified: ppmi_mofa_outputs/model_seed_3.hdf5 (ELBO: -5111020.391811946)

In [ ]:
python3 scripts/extract_factors.py \
  --model ppmi_mofa_outputs/model_seed_3.hdf5 \
  --metadata ppmi_mofa_inputs/ppmi_master_metadata_mofa_ready.csv \
  --output ppmi_mofa_outputs \
  --top_pct 0.05

#Loading trained model from: ppmi_mofa_outputs/model_seed_3.hdf5
#Views: ['CSF_Proteomics', 'Plasma_Proteomics', 'RNA_seq']
#
#=== 1. Extracting Variance Explained (R2) ===
#Saved variance explained per factor to: ppmi_mofa_outputs/variance_explained_per_factor.csv
#Saved variance explained bar plot to: ppmi_mofa_outputs/variance_explained_bar_plot.png
#Saved variance explained heatmap to: ppmi_mofa_outputs/variance_explained_heatmap.png
#Saved total variance explained per view and group to: ppmi_mofa_outputs/variance_explained_total.csv
#
#=== 2. Extracting Latent Factors (Z Matrix) ===
#Merging latent factors with metadata from: ppmi_mofa_inputs/ppmi_master_metadata_mofa_ready.csv
#Saved latent factors Z to: ppmi_mofa_outputs/latent_factors_Z_with_meta.csv
#
#=== 3. Extracting Feature Weights (W Matrix) ===
#Saved full feature weights to: ppmi_mofa_outputs/feature_weights_W_full.csv
#Saved top 5% feature drivers to: ppmi_mofa_outputs/top_5pct_feature_weights.csv
#
#=== Extraction Complete ===

In [ ]:
python3 scripts/extract_factors.py \
  --model ppmi_mofa_outputs/model_seed_3.hdf5 \
  --metadata ppmi_mofa_inputs/ppmi_master_metadata_mofa_ready.csv \
  --output ppmi_mofa_outputs \
  --top_pct 0.05

# OUTPUT:
#Loading trained model from: ppmi_mofa_outputs/model_seed_3.hdf5
#
#=== 1. Extracting Variance Explained (R2) ===
#Saved variance explained per factor to: ppmi_mofa_outputs/variance_explained_per_factor.csv
#Saved variance explained bar plot to: ppmi_mofa_outputs/variance_explained_bar_plot.png
#Saved variance explained heatmap to: ppmi_mofa_outputs/variance_explained_heatmap.png
#
#=== 2. Extracting Latent Factors (Z Matrix) ===
#Merging latent factors with metadata from: ppmi_mofa_inputs/ppmi_master_metadata_mofa_ready.csv
#Saved latent factors Z to: ppmi_mofa_outputs/latent_factors_Z_with_meta.csv
#
#=== 3. Extracting Feature Weights (W Matrix) ===
#Saved full feature weights to: ppmi_mofa_outputs/feature_weights_W_full.csv
#Saved top 5% feature drivers to: ppmi_mofa_outputs/top_5pct_feature_weights.csv
#
#=== Extraction Complete ===

In [ ]:
python3 scripts/evaluate_factors.py \
  --input_z ppmi_mofa_outputs/latent_factors_Z_with_meta.csv \
  --metadata data/metadata_m0_PPMI.csv \
  --output ppmi_mofa_outputs \
  --covariates diagnosis sex age_at_baseline plate mds_updrs_part_iii_summary_score moca_total_score

# OUTPUT
#ID Namespace Check: 1329 IDs in Z matrix, 1394 IDs in metadata. Overlap: 1329
#Merged latent factors with metadata: 1432 samples matched.
#Found latent factors: ['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5', 'Factor6', 'Factor7', 'Factor8', 'Factor9', 'Factor10']
#Evaluating associations against: ['diagnosis', 'sex', 'age_at_baseline', 'plate', 'mds_updrs_part_iii_summary_score', 'moca_total_score']
#Saved association statistics to: ppmi_mofa_outputs/factor_metadata_associations.csv
#Saved association heatmap to: ppmi_mofa_outputs/factor_metadata_associations_heatmap.png
#
#=== Evaluation Complete ===

# PDBP

## Preprocessing

### Prepare metadata

#### Add clinical measures for association analysis

In [ ]:
# Checkpoint
meta_pdbp = pd.read_csv('../data/metadata_m0_PDBP.csv')

### Prepare expression data

In [ ]:
df = pd.read_csv('../data/pdbp/raw/pdbp_rnaseq.csv', index_col=0)                                                                            
np.log1p(df).to_csv('../data/pdbp/pdbp_rnaseq_log1p.csv')    

In [ ]:
! ./adjust_batch_effects.R \
--expression ../data/pdbp/pdbp_rnaseq_log1p.csv \
--metadata ../data/metadata_m0_PDBP.csv \
--output ../data/pdbp_rnaseq_adjusted.csv \
--plot_prefix ../figures/pdbp_rnaseq_pca

In [ ]:
python3 scripts/preprocess_pdbp.py \
  --rna_seq data/pdbp_rnaseq_adjusted.csv \
  --csf_proteomics data/pdbp/raw/csf_proteomics.csv \
  --plasma_proteomics data/pdbp/raw/plasma_proteomics.csv \
  --model_path ppmi_mofa_outputs/model_seed_3.hdf5 \
  --output_dir pdbp_mofa_inputs \
  --metadata_path data/metadata_m0_PDBP.csv

# OUTPUT
#Starting PDBP preprocessing for cohort: PDBP
#Loading metadata from data/metadata_m0_PDBP.csv...
#Processing RNA_seq from data/pdbp_rnaseq_adjusted.csv...
#PDBP RNA_seq: Log-transformed/normalized range detected. Skipping log1p transformation...Saved aligned RNA_seq matrix to pdbp_mofa_inputs/pdbp_rna_seq_aligned.csv (Shape: (5000, 1248))
#Processing CSF_Proteomics from data/pdbp/raw/csf_proteomics.csv...
#Saved aligned CSF_Proteomics matrix to pdbp_mofa_inputs/pdbp_csf_proteomics_aligned.csv (Shape: (249, 107))
#Processing Plasma_Proteomics from data/pdbp/raw/plasma_proteomics.csv...
#Saved aligned Plasma_Proteomics matrix to pdbp_mofa_inputs/pdbp_plasma_proteomics_aligned.csv (Shape: (202, 107))
#Saved long-format matrix to pdbp_mofa_inputs/pdbp_mofa_input_long.csv (Shape: (6288257, 5))
#
#Preprocessing complete.

In [ ]:
# Verify that data was processed correctly
import h5py
import pandas as pd
import numpy as np

model_path = "../ppmi_mofa_outputs/model_seed_3.hdf5"
views = {
    "RNA_seq": "../pdbp_mofa_inputs/pdbp_rna_seq_aligned.csv",
    "CSF_Proteomics": "../pdbp_mofa_inputs/pdbp_csf_proteomics_aligned.csv",
    "Plasma_Proteomics": "../pdbp_mofa_inputs/pdbp_plasma_proteomics_aligned.csv"
}

print("=== 1. Feature Order & Count Alignment ===")
with h5py.File(model_path, 'r') as f:
    for view, file_path in views.items():
        # Get model features
        model_feats = [x.decode('utf-8') if isinstance(x, bytes) else str(x) 
                       for x in f['features'][view][:]]
        
        # Load aligned matrix
        df = pd.read_csv(file_path, index_col=0)
        file_feats = list(df.index)
        
        # Check alignment
        is_exact_match = (model_feats == file_feats)
        print(f"[{view}] Model features: {len(model_feats)} | File features: {len(file_feats)} | Perfect Match: {is_exact_match}")

print("\n=== 2. Missing Value (NaN) & Scaling Check ===")
for view, file_path in views.items():
    df = pd.read_csv(file_path, index_col=0)
    
    nan_pct = (df.isna().sum().sum() / df.size) * 100
    mean_val = df.mean().mean()
    min_val, max_val = df.min().min(), df.max().max()
    
    print(f"[{view}] Missing values: {nan_pct:.2f}% | Mean: {mean_val:.3f} | Min: {min_val:.3f} | Max: {max_val:.3f}")

print("\n=== 3. Sample ID Consistency Check ===")
rnaseq_samples = set(pd.read_csv(views["RNA_seq"], index_col=0).columns)
csf_samples = set(pd.read_csv(views["CSF_Proteomics"], index_col=0).columns)
plasma_samples = set(pd.read_csv(views["Plasma_Proteomics"], index_col=0).columns)

print(f"Total RNA-seq samples: {len(rnaseq_samples)}")
print(f"Total CSF samples: {len(csf_samples)}")
print(f"Total Plasma samples: {len(plasma_samples)}")
print(f"Samples in CSF & Plasma: {len(csf_samples.intersection(plasma_samples))}")
print(f"Samples in RNA-seq & CSF: {len(rnaseq_samples.intersection(csf_samples))}")

## Project factors

In [ ]:
# 1. Project PDBP and generate true PDBP R2 CSV (Run in terminal)
python3 scripts/project_pdbp.py \
  --model_path ppmi_mofa_outputs/model_seed_3.hdf5 \
  --input_dir pdbp_mofa_inputs/ \
  --output_dir pdbp_mofa_outputs/ \
  --metadata data/metadata_m0_PDBP.csv \
  --cohort PDBP

# OUTPUT:
#Loading trained weights from: ppmi_mofa_outputs/model_seed_3.hdf5
#Model groups: ['Female', 'Male']
#View RNA_seq: 5000 / 5000 model features observed in PDBP.
#View CSF_Proteomics: 249 / 249 model features observed in PDBP.
#View Plasma_Proteomics: 202 / 202 model features observed in PDBP.
#Saved projected variance explained R2 to: pdbp_mofa_outputs/pdbp_variance_explained_per_factor.csv
#Saved projected total variance explained to: pdbp_mofa_outputs/pdbp_variance_explained_total.csv
#
#Successfully projected 1248 samples into 10 MOFA factors.
#Saved projected factors Z to: pdbp_mofa_outputs/pdbp_latent_factors_Z.csv

In [ ]:
# 2. Evaluate factor associations and compare PPMI vs PDBP R2 replication (Run in terminal)
python3 scripts/evaluate_factors.py \
  --input_z pdbp_mofa_outputs/pdbp_latent_factors_Z.csv \
  --metadata data/metadata_m0_PDBP.csv \
  --ppmi_r2 ppmi_mofa_outputs/variance_explained_per_factor.csv \
  --pdbp_r2 pdbp_mofa_outputs/pdbp_variance_explained_per_factor.csv \
  --output pdbp_mofa_outputs \
  --covariates diagnosis sex age_at_baseline plate mds_updrs_part_iii_summary_score moca_total_score

# OUTPUT:
#ID Namespace Check: 1248 IDs in Z matrix, 1376 IDs in metadata. Overlap: 1248
#Merged latent factors with metadata: 1248 samples matched.
#Found latent factors: ['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5', 'Factor6', 'Factor7', 'Factor8', 'Factor9', 'Factor10']
#Evaluating associations against: ['diagnosis', 'sex', 'age_at_baseline', 'plate', 'mds_updrs_part_iii_summary_score', 'moca_total_score']
#Saved association statistics to: pdbp_mofa_outputs/factor_metadata_associations.csv
#Saved association heatmap to: pdbp_mofa_outputs/factor_metadata_associations_heatmap.png
#Comparing variance explained ($R^2$) replication across cohorts...
#Saved replication comparison table to: pdbp_mofa_outputs/cohort_variance_replication_comparison.csv
#  CSF_Proteomics / Female: Spearman rho of per-factor R2 = 0.12
#  CSF_Proteomics / Male: Spearman rho of per-factor R2 = 0.33
#  Plasma_Proteomics / Female: Spearman rho of per-factor R2 = -0.79
#  Plasma_Proteomics / Male: Spearman rho of per-factor R2 = -0.88
#  RNA_seq / Female: Spearman rho of per-factor R2 = 0.96
#  RNA_seq / Male: Spearman rho of per-factor R2 = 0.95
#Saved replication comparison plot to: pdbp_mofa_outputs/cohort_variance_replication_comparison.png
#
#=== Evaluation Complete ===

# QC

In [ ]:
python3 scripts/plasma_diagnostics.py \
  --ppmi_plasma data/ppmi/raw/ppmi_pla_prot.csv \
  --pdbp_plasma data/pdbp/raw/pdbp_pla_prot.csv \
  --ppmi_csf data/ppmi/raw/ppmi_csf_prot.csv \
  --pdbp_csf data/pdbp/raw/pdbp_csf_prot.csv \
  --ppmi_metadata data/metadata_m0_PPMI.csv \
  --ppmi_z ppmi_mofa_outputs/latent_factors_Z_with_meta.csv \
  --output plasma_diagnostics

- Proteomics dominated by zeros (strongly in plasma, less so in CSF)
- In PPMI:
    - Filter out proteins w/ >30% zeros (45/202)
    - Drop samples w/ >50% zeros as QC failure
    - PDBP then keeps filtered PPMI protein list